# Diffusion Inpainting Benchmark on COCO val2017

Цель проекта: оценить, как diffusion inpainting модели восстанавливают фон после удаления объекта с изображения. Для каждой задачи используется COCO instance mask, поэтому известны и область удаления, и исходное изображение с ground truth.

Работа построена как benchmark: параметры подбираются на отдельном `tune` split, а финальное сравнение моделей проводится на отложенном `test` split. Это позволяет не подбирать настройки на тех же примерах, на которых затем измеряется качество.

## Литература

- Rombach et al., *High-Resolution Image Synthesis with Latent Diffusion Models*, CVPR 2022. https://openaccess.thecvf.com/content/CVPR2022/html/Rombach_High-Resolution_Image_Synthesis_With_Latent_Diffusion_Models_CVPR_2022_paper.html
- Lugmayr et al., *RePaint: Inpainting Using Denoising Diffusion Probabilistic Models*, CVPR 2022. https://openaccess.thecvf.com/content/CVPR2022/html/Lugmayr_RePaint_Inpainting_Using_Denoising_Diffusion_Probabilistic_Models_CVPR_2022_paper.html
- Kirillov et al., *Segment Anything*, ICCV 2023. https://openaccess.thecvf.com/content/ICCV2023/html/Kirillov_Segment_Anything_ICCV_2023_paper.html

Latent Diffusion Models задают основу для Stable Diffusion. RePaint мотивирует постановку inpainting как восстановление неизвестной области при сохранении наблюдаемого контекста. COCO masks выступают как воспроизводимый источник областей удаления.

## Гипотезы

In [ ]:
HYPOTHESES = {
    "H1": "SDXL Inpainting даст лучше perceptual quality по ROI LPIPS, но будет заметно медленнее SD 1.5.",
    "H2": "Увеличение dilation mask улучшит стык на границе объекта, но ухудшит PSNR из-за большей неизвестной области.",
    "H3": "Параметры, выбранные на tune split, не обязаны быть лучшими на test split, поэтому splits разделены.",
}

for key, text in HYPOTHESES.items():
    print(f"{key}: {text}")

## Установка

In [ ]:
import subprocess
import sys

packages = [
    "diffusers==0.36.0",
    "transformers==4.57.3",
    "accelerate>=1.0.0,<2.0",
    "huggingface_hub>=0.34.0,<1.0.0",
    "safetensors>=0.4.0",
    "Pillow==11.3.0",
    "numpy>=1.26,<2.1",
    "pandas==2.2.2",
    "requests==2.32.4",
    "scikit-image==0.25.2",
    "lpips==0.1.4",
    "matplotlib>=3.8,<3.11",
    "tqdm>=4.66,<5.0",
    "pycocotools",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--upgrade-strategy", "only-if-needed", *packages],
    check=True,
)

print("Installation completed. Restart session now, then continue with Imports.")

## Импорты

In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import time
import zipfile
from dataclasses import dataclass, asdict
from pathlib import Path

import cv2
import diffusers
import lpips
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL
import requests
import skimage
import torch

from PIL import Image
from pycocotools.coco import COCO
from skimage.metrics import structural_similarity
from tqdm.auto import tqdm

from transformers import AutoImageProcessor

from diffusers import (
    KandinskyV22InpaintPipeline,
    KandinskyV22PriorPipeline,
    StableDiffusionInpaintPipeline,
    StableDiffusionXLInpaintPipeline,
)

## Preflight

In [ ]:
def package_version(module):
    return getattr(module, "__version__", "unknown")

import subprocess
import transformers

print("torch:", torch.__version__)
print("diffusers:", package_version(diffusers))
print("transformers:", package_version(transformers))
print("Pillow:", PIL.__version__)
print("scikit-image:", package_version(skimage))
print("CUDA available:", torch.cuda.is_available())

if not hasattr(transformers, "AutoImageProcessor"):
    raise RuntimeError("AutoImageProcessor is missing. Run the installation cell, restart session and retry.")

if diffusers.__version__ != "0.36.0":
    raise RuntimeError(f"Expected diffusers 0.36.0, got {diffusers.__version__}. Restart session after installation.")

check = subprocess.run(["pip", "check"], capture_output=True, text=True)
if check.returncode != 0:
    raise RuntimeError("Dependency check failed. Restart Colab session and rerun installation.\n" + check.stdout)

if not torch.cuda.is_available():
    raise RuntimeError("Для benchmark нужен GPU runtime в Google Colab.")

device = "cuda"
gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name}, memory: {gpu_mem_gb:.1f} GB")

if gpu_mem_gb < 12:
    print("Warning: SDXL будет работать медленно. Для него включен CPU offload.")
else:
    print("GPU memory is sufficient for sequential SDXL inference with CPU offload.")

## Конфигурация

In [ ]:
ROOT = Path(".")
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs"
IMAGE_DIR = DATA_DIR / "val2017"
DATA_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

SEED = 42
BENCH_MODE = "research"  # "debug", "research", "full"

MODE_CONFIG = {
    "debug": {"per_category": 1, "max_tasks": 40, "tune": 10, "test": 30},
    "research": {"per_category": 12, "max_tasks": 500, "tune": 100, "test": 400},
    "full": {"per_category": 80, "max_tasks": 5000, "tune": 1000, "test": 4000},
}
CFG = MODE_CONFIG[BENCH_MODE]

MIN_MASK_AREA = 2_500
MAX_MASK_AREA_RATIO = 0.35
MAX_IMAGE_SIDE = 512
DOWNLOAD_FULL_VAL2017 = BENCH_MODE in {"research", "full"}
SAVE_PREVIEWS = 24

BASE_PROMPT = "a realistic high quality photograph"
PROMPT_STYLES = {
    "neutral": lambda category: (BASE_PROMPT, category),
    "background": lambda category: (f"a realistic photo of the background without a {category}", category),
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("mode:", BENCH_MODE)
print("config:", CFG)

## COCO аннотации

In [ ]:
def download_file(url, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        return path

    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()
    total = int(response.headers.get("content-length", 0))
    with open(path, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=path.name) as bar:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))
    return path

COCO_ANN_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
COCO_ANN_ZIP = DATA_DIR / "annotations_trainval2017.zip"
COCO_ANN_PATH = DATA_DIR / "annotations" / "instances_val2017.json"

if not COCO_ANN_PATH.exists():
    download_file(COCO_ANN_URL, COCO_ANN_ZIP)
    with zipfile.ZipFile(COCO_ANN_ZIP, "r") as archive:
        archive.extract("annotations/instances_val2017.json", DATA_DIR)

coco = COCO(str(COCO_ANN_PATH))
print("COCO validation images:", len(coco.getImgIds()))
print("COCO categories:", len(coco.getCatIds()))

## COCO изображения

In [ ]:
COCO_VAL_URL = "http://images.cocodataset.org/zips/val2017.zip"
COCO_VAL_ZIP = DATA_DIR / "val2017.zip"

if DOWNLOAD_FULL_VAL2017 and not IMAGE_DIR.exists():
    download_file(COCO_VAL_URL, COCO_VAL_ZIP)
    with zipfile.ZipFile(COCO_VAL_ZIP, "r") as archive:
        archive.extractall(DATA_DIR)

IMAGE_CACHE = {}

def get_coco_image(image_id):
    image_id = int(image_id)
    if image_id in IMAGE_CACHE:
        return IMAGE_CACHE[image_id]

    info = coco.loadImgs([image_id])[0]
    local_path = IMAGE_DIR / info["file_name"]

    if not local_path.exists():
        url = info.get("coco_url") or f"http://images.cocodataset.org/val2017/{info['file_name']}"
        download_file(url, local_path)

    image = Image.open(local_path).convert("RGB")
    IMAGE_CACHE[image_id] = (image, info)
    return image, info

## Генерация задач

In [ ]:
def build_task_pool(per_category, max_tasks, min_area, max_area_ratio, seed):
    rng = random.Random(seed)
    category_tasks = []

    categories = sorted(coco.loadCats(coco.getCatIds()), key=lambda item: item["name"])
    for category in tqdm(categories, desc="categories"):
        cat_id = category["id"]
        candidates = []

        for image_id in coco.getImgIds(catIds=[cat_id]):
            image_info = coco.loadImgs([image_id])[0]
            image_area = image_info["width"] * image_info["height"]
            ann_ids = coco.getAnnIds(imgIds=[image_id], catIds=[cat_id], iscrowd=False)
            anns = coco.loadAnns(ann_ids)

            valid = [
                ann for ann in anns
                if ann.get("area", 0) >= min_area and ann.get("area", 0) / image_area <= max_area_ratio
            ]
            if not valid:
                continue

            target = max(valid, key=lambda ann: ann["area"])
            candidates.append(
                {
                    "image_id": int(image_id),
                    "ann_id": int(target["id"]),
                    "category_id": int(cat_id),
                    "category": category["name"],
                    "mask_area": float(target["area"]),
                    "mask_area_ratio": float(target["area"] / image_area),
                }
            )

        rng.shuffle(candidates)
        category_tasks.append(candidates[:per_category])

    tasks = []
    max_per_category = max((len(items) for items in category_tasks), default=0)
    for index in range(max_per_category):
        for items in category_tasks:
            if index < len(items):
                tasks.append(items[index])
                if len(tasks) >= max_tasks:
                    return pd.DataFrame(tasks)

    return pd.DataFrame(tasks)

tasks_df = build_task_pool(
    per_category=CFG["per_category"],
    max_tasks=CFG["max_tasks"],
    min_area=MIN_MASK_AREA,
    max_area_ratio=MAX_MASK_AREA_RATIO,
    seed=SEED,
)

tasks_df.to_csv(OUT_DIR / "coco_task_pool.csv", index=False)
print("tasks:", len(tasks_df))
print("categories:", tasks_df["category"].nunique())
tasks_df.head()

## Tune test split

In [ ]:
def split_tasks(df, tune_size, test_size, seed):
    shuffled = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    tune = shuffled.iloc[:tune_size].copy()
    test = shuffled.iloc[tune_size:tune_size + test_size].copy()

    if len(test) != test_size:
        raise ValueError("Недостаточно задач. Увеличь max_tasks или уменьшай split sizes.")

    tune["split"] = "tune"
    test["split"] = "test"
    return tune.reset_index(drop=True), test.reset_index(drop=True)

tune_tasks, test_tasks = split_tasks(tasks_df, CFG["tune"], CFG["test"], SEED)
all_splits = pd.concat([tune_tasks, test_tasks], ignore_index=True)
all_splits.to_csv(OUT_DIR / "coco_tune_test_split.csv", index=False)

print("tune:", len(tune_tasks), "tasks /", tune_tasks["category"].nunique(), "categories")
print("test:", len(test_tasks), "tasks /", test_tasks["category"].nunique(), "categories")

## Маски и входные данные

In [ ]:
def ann_mask(ann_id):
    ann = coco.loadAnns([int(ann_id)])[0]
    return coco.annToMask(ann).astype(np.uint8)

def dilate_mask(mask, pixels):
    if pixels <= 0:
        return mask
    size = pixels * 2 + 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))
    return cv2.dilate(mask, kernel, iterations=1)

def resize_image_and_mask(image, mask, max_side=512):
    width, height = image.size
    scale = min(1.0, max_side / max(width, height))
    new_w = max(8, int(round(width * scale / 8) * 8))
    new_h = max(8, int(round(height * scale / 8) * 8))

    image = image.resize((new_w, new_h), Image.Resampling.LANCZOS)
    mask_img = Image.fromarray((mask > 0).astype(np.uint8) * 255)
    mask_img = mask_img.resize((new_w, new_h), Image.Resampling.NEAREST)
    return image, mask_img

def make_masked_image(image, mask_img, fill=127):
    arr = np.array(image).copy()
    mask = np.array(mask_img) > 0
    arr[mask] = fill
    return Image.fromarray(arr)

def prepare_task(row, dilation_px):
    image, _ = get_coco_image(row["image_id"])
    original_mask = ann_mask(row["ann_id"])
    dilated = dilate_mask(original_mask, dilation_px)
    reference, mask_img = resize_image_and_mask(image, dilated, MAX_IMAGE_SIDE)
    masked = make_masked_image(reference, mask_img)
    return reference, masked, mask_img

## Dataset sanity check

In [ ]:
row = tune_tasks.iloc[0]
reference, masked, mask_img = prepare_task(row, dilation_px=8)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(reference)
axes[0].set_title("reference")
axes[1].imshow(mask_img, cmap="gray")
axes[1].set_title(f"mask: {row['category']}")
axes[2].imshow(masked)
axes[2].set_title("inpainting input")
for axis in axes:
    axis.axis("off")
plt.show()

## Модели

In [ ]:
@dataclass(frozen=True)
class ModelSpec:
    key: str
    model_id: str
    pipeline_class: str
    prior_model_id: str | None = None

MODEL_SPECS = {
    "sd15_inpaint": ModelSpec(
        key="sd15_inpaint",
        model_id="runwayml/stable-diffusion-inpainting",
        pipeline_class="StableDiffusionInpaintPipeline",
    ),
    "sdxl_inpaint": ModelSpec(
        key="sdxl_inpaint",
        model_id="diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
        pipeline_class="StableDiffusionXLInpaintPipeline",
    ),
    "kandinsky22_inpaint": ModelSpec(
        key="kandinsky22_inpaint",
        model_id="kandinsky-community/kandinsky-2-2-decoder-inpaint",
        pipeline_class="KandinskyV22InpaintPipeline",
        prior_model_id="kandinsky-community/kandinsky-2-2-prior",
    ),
}

TORCH_DTYPE = torch.float16

## Загрузка pipeline

In [ ]:
def load_pipeline(model_key):
    spec = MODEL_SPECS[model_key]
    if spec.pipeline_class == "KandinskyV22InpaintPipeline":
        prior = KandinskyV22PriorPipeline.from_pretrained(spec.prior_model_id, torch_dtype=TORCH_DTYPE)
        decoder = KandinskyV22InpaintPipeline.from_pretrained(spec.model_id, torch_dtype=TORCH_DTYPE)
        prior.enable_model_cpu_offload()
        decoder.enable_model_cpu_offload()
        return {"kind": "kandinsky", "prior": prior, "decoder": decoder}

    cls = StableDiffusionInpaintPipeline if spec.pipeline_class == "StableDiffusionInpaintPipeline" else StableDiffusionXLInpaintPipeline

    try:
        pipe = cls.from_pretrained(spec.model_id, torch_dtype=TORCH_DTYPE, use_safetensors=True)
    except OSError:
        pipe = cls.from_pretrained(spec.model_id, torch_dtype=TORCH_DTYPE)

    pipe.enable_attention_slicing()
    pipe.enable_vae_slicing()
    pipe.enable_model_cpu_offload()
    return pipe

def unload_pipeline(pipe):
    if isinstance(pipe, dict):
        del pipe["prior"]
        del pipe["decoder"]
    else:
        del pipe
    gc.collect()
    torch.cuda.empty_cache()

## Метрики

In [ ]:
def to_float_array(image):
    return np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0

def mask_array(mask_img):
    return np.asarray(mask_img, dtype=np.uint8) > 0

def roi_bounds(mask, padding=32):
    ys, xs = np.where(mask)
    y0 = max(0, int(ys.min()) - padding)
    y1 = min(mask.shape[0], int(ys.max()) + padding + 1)
    x0 = max(0, int(xs.min()) - padding)
    x1 = min(mask.shape[1], int(xs.max()) + padding + 1)
    return y0, y1, x0, x1

def masked_psnr(pred, target, mask):
    mse = ((pred[mask] - target[mask]) ** 2).mean()
    return float(99.0 if mse < 1e-12 else 10 * np.log10(1.0 / mse))

def roi_ssim(pred, target, mask):
    y0, y1, x0, x1 = roi_bounds(mask)
    return float(
        structural_similarity(
            target[y0:y1, x0:x1],
            pred[y0:y1, x0:x1],
            channel_axis=2,
            data_range=1.0,
        )
    )

LPIPS_MODEL = None

def get_lpips_model():
    global LPIPS_MODEL
    if LPIPS_MODEL is None:
        LPIPS_MODEL = lpips.LPIPS(net="alex").to(device).eval()
    return LPIPS_MODEL

def roi_lpips(pred, target, mask):
    y0, y1, x0, x1 = roi_bounds(mask)
    pred_roi = torch.from_numpy(pred[y0:y1, x0:x1]).permute(2, 0, 1).unsqueeze(0).to(device)
    target_roi = torch.from_numpy(target[y0:y1, x0:x1]).permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.inference_mode():
        value = get_lpips_model()(pred_roi * 2 - 1, target_roi * 2 - 1)
    return float(value.item())

def outside_mae(pred, target, mask):
    return float(np.abs(pred[~mask] - target[~mask]).mean())

def compute_metrics(generated, reference, mask_img):
    pred = to_float_array(generated)
    target = to_float_array(reference)
    mask = mask_array(mask_img)
    return {
        "masked_psnr": masked_psnr(pred, target, mask),
        "roi_ssim": roi_ssim(pred, target, mask),
        "roi_lpips": roi_lpips(pred, target, mask),
        "outside_mae": outside_mae(pred, target, mask),
    }

## Tuning grid

In [ ]:
TUNING_GRID = [
    {"steps": 25, "guidance": 4.5, "dilation": 8, "prompt_style": "neutral"},
    {"steps": 25, "guidance": 7.5, "dilation": 8, "prompt_style": "neutral"},
    {"steps": 40, "guidance": 4.5, "dilation": 8, "prompt_style": "neutral"},
    {"steps": 40, "guidance": 7.5, "dilation": 8, "prompt_style": "neutral"},
    {"steps": 25, "guidance": 4.5, "dilation": 16, "prompt_style": "background"},
    {"steps": 25, "guidance": 7.5, "dilation": 16, "prompt_style": "background"},
    {"steps": 40, "guidance": 4.5, "dilation": 16, "prompt_style": "background"},
    {"steps": 40, "guidance": 7.5, "dilation": 16, "prompt_style": "background"},
]

pd.DataFrame(TUNING_GRID)

## Inference runner

In [ ]:
RESULTS_PATH = OUT_DIR / "all_results.csv"
PREVIEW_DIR = OUT_DIR / "previews"
PREVIEW_DIR.mkdir(exist_ok=True)

def run_id(model_key, config, row):
    payload = {
        "model": model_key,
        "config": config,
        "image_id": int(row["image_id"]),
        "ann_id": int(row["ann_id"]),
        "seed": SEED + int(row["ann_id"]),
    }
    return hashlib.sha1(json.dumps(payload, sort_keys=True).encode()).hexdigest()

def load_existing_results():
    if RESULTS_PATH.exists():
        return pd.read_csv(RESULTS_PATH)
    return pd.DataFrame()

def prompt_for(category, style):
    prompt, negative = PROMPT_STYLES[style](category)
    return prompt, negative

def infer_one(pipe, row, config):
    reference, masked, mask_img = prepare_task(row, dilation_px=config["dilation"])
    prompt, negative_prompt = prompt_for(row["category"], config["prompt_style"])
    seed = SEED + int(row["ann_id"])
    generator = torch.Generator(device=device).manual_seed(seed)

    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=TORCH_DTYPE):
        if isinstance(pipe, dict) and pipe["kind"] == "kandinsky":
            prior_result = pipe["prior"](
                prompt=prompt,
                negative_prompt=negative_prompt,
                num_inference_steps=25,
                guidance_scale=4.0,
                generator=generator,
            )
            result = pipe["decoder"](
                image_embeds=prior_result.image_embeds,
                negative_image_embeds=prior_result.negative_image_embeds,
                image=masked,
                mask_image=mask_img,
                num_inference_steps=config["steps"],
                guidance_scale=config["guidance"],
                generator=generator,
            )
        else:
            result = pipe(
                prompt=prompt,
                negative_prompt=negative_prompt,
                image=masked,
                mask_image=mask_img,
                num_inference_steps=config["steps"],
                guidance_scale=config["guidance"],
                generator=generator,
            )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    generated = result.images[0].convert("RGB")

    metrics = compute_metrics(generated, reference, mask_img)
    return generated, reference, masked, mask_img, elapsed, metrics

def evaluate_config(tasks, model_key, config, split, save_previews=False):
    existing = load_existing_results()
    done = set(existing["run_id"].astype(str)) if len(existing) else set()
    new_rows = []
    pipe = load_pipeline(model_key)

    try:
        for task_index, (_, row) in enumerate(tqdm(tasks.iterrows(), total=len(tasks), desc=f"{model_key} {split}")):
            row = row.to_dict()
            key = run_id(model_key, config, row)
            if key in done:
                continue

            generated, reference, masked, mask_img, elapsed, metrics = infer_one(pipe, row, config)
            preview_path = ""

            if save_previews and task_index < SAVE_PREVIEWS:
                preview_path = PREVIEW_DIR / f"{model_key}_{split}_{key}.png"
                canvas = Image.new("RGB", (reference.width * 3, reference.height))
                canvas.paste(reference, (0, 0))
                canvas.paste(masked, (reference.width, 0))
                canvas.paste(generated, (reference.width * 2, 0))
                canvas.save(preview_path)

            record = {
                "run_id": key,
                "model": model_key,
                "split": split,
                "image_id": int(row["image_id"]),
                "ann_id": int(row["ann_id"]),
                "category": row["category"],
                "mask_area": float(row["mask_area"]),
                "mask_area_ratio": float(row["mask_area_ratio"]),
                "steps": config["steps"],
                "guidance": config["guidance"],
                "dilation": config["dilation"],
                "prompt_style": config["prompt_style"],
                "seed": SEED + int(row["ann_id"]),
                "time_sec": elapsed,
                "preview": str(preview_path),
                **metrics,
            }
            new_rows.append(record)

            if len(new_rows) % 5 == 0:
                combined = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
                combined.to_csv(RESULTS_PATH, index=False)

        if new_rows:
            combined = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
            combined.to_csv(RESULTS_PATH, index=False)
            return combined
        return existing
    finally:
        unload_pipeline(pipe)

## Эксперимент 1: tuning

In [ ]:
for config in TUNING_GRID:
    evaluate_config(
        tune_tasks,
        model_key="sd15_inpaint",
        config=config,
        split="tune",
        save_previews=False,
    )

## Выбор конфигурации

In [ ]:
results_df = load_existing_results()
tune_results = results_df[(results_df["split"] == "tune") & (results_df["model"] == "sd15_inpaint")].copy()

tune_summary = (
    tune_results.groupby(["steps", "guidance", "dilation", "prompt_style"], as_index=False)
    .agg(
        tasks=("run_id", "count"),
        masked_psnr=("masked_psnr", "mean"),
        roi_ssim=("roi_ssim", "mean"),
        roi_lpips=("roi_lpips", "mean"),
        outside_mae=("outside_mae", "mean"),
        time_sec=("time_sec", "mean"),
    )
)

# Primary metric: perceptual quality. Tie-breakers: structural quality and preservation outside mask.
tune_summary = tune_summary.sort_values(
    ["roi_lpips", "roi_ssim", "outside_mae"],
    ascending=[True, False, True],
).reset_index(drop=True)

tune_summary.to_csv(OUT_DIR / "tune_summary.csv", index=False)
tune_summary

## Best configuration

In [ ]:
best_config = tune_summary.iloc[0][["steps", "guidance", "dilation", "prompt_style"]].to_dict()
best_config["steps"] = int(best_config["steps"])
best_config["guidance"] = float(best_config["guidance"])
best_config["dilation"] = int(best_config["dilation"])

print("best configuration:", best_config)
print("selection metric: lowest ROI LPIPS on tune split")

## Эксперимент 2: final test

In [ ]:
for model_key in ["sd15_inpaint", "sdxl_inpaint", "kandinsky22_inpaint"]:
    evaluate_config(
        test_tasks,
        model_key=model_key,
        config=best_config,
        split="test",
        save_previews=True,
    )

## Final metrics

In [ ]:
results_df = load_existing_results()
test_results = results_df[results_df["split"] == "test"].copy()

final_summary = (
    test_results.groupby("model", as_index=False)
    .agg(
        tasks=("run_id", "count"),
        categories=("category", "nunique"),
        masked_psnr=("masked_psnr", "mean"),
        roi_ssim=("roi_ssim", "mean"),
        roi_lpips=("roi_lpips", "mean"),
        outside_mae=("outside_mae", "mean"),
        time_sec=("time_sec", "mean"),
    )
    .sort_values("roi_lpips")
)

final_summary.to_csv(OUT_DIR / "final_test_summary.csv", index=False)
final_summary

## Metric plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].bar(final_summary["model"], final_summary["masked_psnr"])
axes[0].set_title("Masked PSNR")
axes[0].set_ylabel("higher is better")

axes[1].bar(final_summary["model"], final_summary["roi_lpips"])
axes[1].set_title("ROI LPIPS")
axes[1].set_ylabel("lower is better")

axes[2].bar(final_summary["model"], final_summary["time_sec"])
axes[2].set_title("Runtime per task")
axes[2].set_ylabel("seconds")

for axis in axes:
    axis.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Интерпретация

In [ ]:
best_perceptual = final_summary.sort_values("roi_lpips").iloc[0]
best_fidelity = final_summary.sort_values("masked_psnr", ascending=False).iloc[0]
fastest = final_summary.sort_values("time_sec").iloc[0]

print(f"Best perceptual quality: {best_perceptual['model']} with ROI LPIPS={best_perceptual['roi_lpips']:.4f}")
print(f"Best pixel fidelity: {best_fidelity['model']} with masked PSNR={best_fidelity['masked_psnr']:.2f}")
print(f"Fastest model: {fastest['model']} with {fastest['time_sec']:.2f}s per task")

if best_perceptual["model"] != best_fidelity["model"]:
    print("Perceptual and pixel metrics prefer different models. This indicates a realism-fidelity tradeoff.")
else:
    print("One model leads both perceptual and pixel metrics on this test split.")

print("The final table is computed only on the held-out COCO test split. Tuning used separate tasks.")

## Category analysis

In [ ]:
category_summary = (
    test_results.groupby(["model", "category"], as_index=False)
    .agg(
        tasks=("run_id", "count"),
        masked_psnr=("masked_psnr", "mean"),
        roi_ssim=("roi_ssim", "mean"),
        roi_lpips=("roi_lpips", "mean"),
        outside_mae=("outside_mae", "mean"),
    )
)

category_summary.to_csv(OUT_DIR / "category_summary.csv", index=False)
category_summary.sort_values("roi_lpips", ascending=False).head(20)

## Preview analysis

In [ ]:
preview_rows = test_results[test_results["preview"].fillna("") != ""].copy()
preview_rows = preview_rows.sort_values("roi_lpips", ascending=False).head(12)
preview_rows[["model", "image_id", "category", "roi_lpips", "masked_psnr", "preview"]]

## Reproducibility

In [ ]:
run_manifest = {
    "bench_mode": BENCH_MODE,
    "mode_config": CFG,
    "seed": SEED,
    "models": {key: asdict(value) for key, value in MODEL_SPECS.items()},
    "best_config": best_config,
    "versions": {
        "torch": torch.__version__,
        "diffusers": diffusers.__version__,
        "Pillow": PIL.__version__,
        "skimage": skimage.__version__,
    },
    "gpu": gpu_name,
}

with open(OUT_DIR / "run_manifest.json", "w", encoding="utf-8") as f:
    json.dump(run_manifest, f, ensure_ascii=False, indent=2)

run_manifest

## Итог

Проект использует COCO val2017 как большой воспроизводимый benchmark. В режиме `research` формируется 500 задач, из которых 100 используются для настройки и 400 для финального сравнения. В режиме `full` доступны 5,000 задач.

Основной результат проекта не сводится к одной визуализации: он показывает, какая модель лучше восстанавливает скрытые области, как параметры влияют на качество, и где возникает компромисс между perceptual realism, pixel fidelity и временем inference.